# Assignment A9: Performance Tuning Deep Dive

Scenario: A PySpark job runs 4 hours; optimize it.
Tasks:
1.      Identify data skew using Spark UI and fix with salting/AQE.
2.      Replace shuffle joins with broadcast joins.
3.      Tune spark.sql.shuffle.partitions.
4.      Cache/persist intermediate DataFrames.
5.      Use explain(True) and interpret Catalyst output.
6.      Document before/after runtime, resources, and DAG.

Deliverable: Markdown report + Spark UI screenshots


## Assignment Objective

A PySpark job is experiencing poor performance due to data skew,
shuffle-heavy joins, inefficient partitioning, and repeated computation.

This notebook demonstrates performance tuning techniques using local
Apache Spark:

- Data skew identification and mitigation
- Salting
- Adaptive Query Execution (AQE)
- Broadcast joins
- Shuffle partition tuning
- Cache and persist
- Catalyst query-plan analysis using `explain(True)`
- Before vs. after performance comparison

The experiments use a synthetic transaction dataset designed to
contain significant data skew.

In [2]:
import time
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Python:", sys.version)
print("PySpark:", __import__("pyspark").__version__)

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
PySpark: 4.2.0


In [3]:
# Create the SparkSession

spark = (
    SparkSession.builder
    .appName("A9_Performance_Tuning")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

KeyboardInterrupt: 

In [1]:
import os
import subprocess

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr)

JAVA_HOME: C:\Program Files\Java\jdk-17.0.2
java version "1.8.0_251"
Java(TM) SE Runtime Environment (build 1.8.0_251-b08)
Java HotSpot(TM) 64-Bit Server VM (build 25.251-b08, mixed mode)



In [2]:
import shutil

print(shutil.which("java"))

C:\Program Files (x86)\Common Files\Oracle\Java\javapath\java.EXE


In [3]:
import os
import shutil

java_home = r"C:\Program Files\Java\jdk-17.0.2"

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + r"\bin;" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("Java executable:", shutil.which("java"))

JAVA_HOME: C:\Program Files\Java\jdk-17.0.2
Java executable: C:\Program Files\Java\jdk-17.0.2\bin\java.EXE


In [5]:
import subprocess

result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)

print(result.stderr)

java version "17.0.2" 2022-01-18 LTS
Java(TM) SE Runtime Environment (build 17.0.2+8-LTS-86)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.2+8-LTS-86, mixed mode, sharing)



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("A9_Performance_Tuning")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.2.0
Spark UI: http://LAPTOP-JUUB6MEL.bbrouter:4040
